# Physiotherapy Chatbot — RAG Trials

A Retrieval-Augmented Generation pipeline built on
**`data/Evidence-physical-therapy-K.-Bo.pdf`**, with its embeddings stored in the
Pinecone index **`physio-chatbot`**.

> Kernel: select **Python (physio-chatbot .venv)** before running.

## Step 1 — Set up & move to the project root
**What it does:** Prints `OK` to confirm the kernel runs, checks the current folder, then moves up one level from `research/` to the project root.

**Why it's needed:** Later cells use the path `"data"`, which only exists at the project root.

In [ ]:
print("OK")

In [ ]:
%pwd

In [ ]:
import os

# Move to the project root so paths like "data" and ".env" resolve.
# Idempotent: only steps up while still inside research/, so re-running is safe.
while os.path.basename(os.getcwd()) == "research":
    os.chdir("..")
print("Working dir:", os.getcwd())

In [ ]:
%pwd

## Step 2 — Load the physio PDF into text
**What it does:** Imports the PDF tools, then reads **only** `Evidence-physical-therapy-K.-Bo.pdf` from `data/` into a list of pages (~435).

**Why it's needed:** We extract the book's text so it can be chunked, searched, and fed to the AI.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# Extract text from ONLY the physiotherapy PDF
def load_physio_pdf(data):
    loader = DirectoryLoader(
        data,
        glob="Evidence-physical-therapy-K.-Bo.pdf",  # this specific book only
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents

In [ ]:
extracted_data = load_physio_pdf("data")

In [ ]:
len(extracted_data)   # number of pages loaded (~435)

## Step 3 — Trim each page to the essentials
**What it does:** Rebuilds each page keeping only its text and `source` filename, dropping the rest of the metadata.

**Why it's needed:** Less clutter keeps the data lightweight and the search results clean.

In [ ]:
from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """Keep only the page text and its 'source' filename."""
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(page_content=doc.page_content, metadata={"source": src})
        )
    return minimal_docs

In [ ]:
minimal_docs = filter_to_minimal_docs(extracted_data)

## Step 4 — Split into searchable chunks
**What it does:** Cuts the pages into ~500-character chunks that overlap by 20 characters.

**Why it's needed:** Small, focused passages give far more precise search hits than whole pages and fit the model's context window.

In [ ]:
# Split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [ ]:
texts_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")

## Step 5 — Turn text into embeddings (meaning vectors)
**What it does:** Loads a local HuggingFace model (`all-MiniLM-L6-v2`) that converts text into 384-number vectors; the test confirms the length is 384.

**Why it's needed:** Embeddings let us measure *meaning* similarity — how we find the chunks most relevant to a question. Runs free, on your machine.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

def download_embeddings():
    """Download and return the HuggingFace embeddings model (384-dim)."""
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return embeddings

embedding = download_embeddings()

In [ ]:
vector = embedding.embed_query("Hello world")
print("Vector length:", len(vector))   # 384

## Step 6 — Load your secret API keys
**What it does:** Reads `PINECONE_API_KEY` and `OPENAI_API_KEY` from the `.env` file into the environment.

**Why it's needed:** Pinecone (storage) and OpenAI (answers) require authentication, and `.env` keeps secrets out of the code.

In [ ]:
from dotenv import load_dotenv
import os

# load_dotenv() reads the .env at the project root and populates os.environ.
# In a notebook it searches the current working directory, so run the Step 1
# cell first. It returns True when a .env was found and loaded.
loaded = load_dotenv()
print("Loaded .env:", loaded)

In [ ]:
# load_dotenv() already put these into os.environ; here we just read them back
# and fail with a clear message if any are missing (instead of a cryptic
# "TypeError: str expected, not NoneType").
required = ["PINECONE_API_KEY", "OPENAI_API_KEY"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(
        f"Missing {missing} in the environment. Add them to the .env file at the "
        "project root, then re-run the 'Load your secret API keys' cell above."
    )

PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
print("API keys loaded ✓")

## Step 7 — Connect to Pinecone
**What it does:** Creates a Pinecone client using your API key.

**Why it's needed:** Pinecone is the cloud vector database where the embeddings are stored and searched.

In [ ]:
from pinecone import Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

## Step 8 — Create the vector index
**What it does:** Creates an index named `physio-chatbot` (only if missing) with `dimension=384` and cosine similarity.

**Why it's needed:** This is the container that holds the embeddings; its dimension must match the embedding size (384) or uploads are rejected.

In [ ]:
from pinecone import ServerlessSpec

index_name = "physio-chatbot"   # the Pinecone index for this chatbot

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,            # must match the embedding size
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

## Step 9 — Upload the chunks (run once)
**What it does:** Embeds every chunk and stores it in the `physio-chatbot` Pinecone index.

**Why it's needed:** This populates the knowledge base. Run it **once** — re-running re-uploads everything. On later runs use Step 10 instead.

In [ ]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

## Step 10 — Reconnect to the existing index
**What it does:** Opens the already-filled index without re-uploading.

**Why it's needed:** On later runs, use this instead of Step 9 to save time and avoid duplicate uploads.

In [ ]:
# Use this on future runs INSTEAD of re-uploading
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

## Step 11 — Build the retriever
**What it does:** Turns the vector store into a retriever returning the top 3 most similar chunks (`k=3`); the test query previews what it finds.

**Why it's needed:** The retriever is the "look-up" half of RAG — it fetches relevant context for each question.

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [ ]:
retrieved_docs = retriever.invoke("What is evidence-based physical therapy?")
retrieved_docs

## Step 12 — Connect the GPT-4o brain
**What it does:** Creates a `ChatOpenAI` model using `gpt-4o`.

**Why it's needed:** This model writes the final natural-language answer (the part that costs money per call).

In [ ]:
from langchain_openai import ChatOpenAI
chatModel = ChatOpenAI(model="gpt-4o")

## Step 13 — Assemble the RAG chain
**What it does:** Defines the physiotherapy system prompt, then wires retriever + model together: retrieve chunks → stuff into the prompt → generate an answer.

**Why it's needed:** This chain *is* the chatbot — "find relevant context, then answer using only it."

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
system_prompt = (
    "You are a Physiotherapy assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [ ]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

## Step 14 — Ask questions
**What it does:** Sends physiotherapy questions through the chain and prints the answers.

**Why it's needed:** The payoff — confirming the bot returns sensible, source-grounded answers from the physio book.

In [ ]:
response = rag_chain.invoke({"input": "What is evidence-based physical therapy?"})
print(response["answer"])

In [ ]:
response = rag_chain.invoke({"input": "What can randomized trials and systematic reviews tell us?"})
print(response["answer"])

In [ ]:
response = rag_chain.invoke({"input": "How should a physiotherapist appraise clinical evidence?"})
print(response["answer"])